# Notebook 06 (Exp 3) — BLEU + BERTScore Evaluation

Evaluates exp3 models. Identical structure to 06_exp2, pointing to `outputs/exp3/` paths.

Also uses `roberta-large` for BERTScore (higher quality than exp2's `distilbert-base-uncased`).

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import AutoPeftModelForCausalLM

sys.path.insert(0, str(Path('..').resolve()))
from src.evaluation import compute_bleu_scores, compute_bert_scores, run_batch_inference

ROOT          = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
LORA_PATH     = ROOT / 'outputs' / 'exp3' / 'lora' / 'final_adapter'
FFT_PATH      = ROOT / 'outputs' / 'exp3' / 'fft'  / 'final_model'
RESULTS_DIR   = ROOT / 'outputs' / 'exp3' / 'results'
FIG_DIR       = ROOT / 'outputs' / 'exp3' / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

TEST_JSONL = PROCESSED_DIR / 'test.jsonl'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
print('Setup complete.')


## 1. LoRA Inference

In [ ]:
print('Loading Exp 3 LoRA model ...')
model_lora = AutoPeftModelForCausalLM.from_pretrained(
    str(LORA_PATH), torch_dtype=torch.bfloat16, device_map='auto',
)
model_lora.eval()
tok_lora = AutoTokenizer.from_pretrained(str(LORA_PATH))

hyps_lora, refs_lora = run_batch_inference(
    model_lora, tok_lora, test_jsonl_path=TEST_JSONL,
    direction='mod2shak', max_new_tokens=256,
)
print(f'Inference complete: {len(hyps_lora)} examples')


In [ ]:
import gc
del model_lora
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('LoRA unloaded.')


## 2. FFT Inference

In [ ]:
print('Loading Exp 3 FFT model ...')
model_fft = AutoModelForCausalLM.from_pretrained(
    str(FFT_PATH), torch_dtype=torch.bfloat16, device_map='auto',
)
model_fft.eval()
tok_fft = AutoTokenizer.from_pretrained(str(FFT_PATH))

hyps_fft, refs_fft = run_batch_inference(
    model_fft, tok_fft, test_jsonl_path=TEST_JSONL,
    direction='mod2shak', max_new_tokens=256,
)
print(f'Inference complete: {len(hyps_fft)} examples')


In [ ]:
del model_fft
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('FFT unloaded.')


## 3. BLEU and ChrF

In [ ]:
scores_lora, sent_bleus_lora = compute_bleu_scores(hyps_lora, refs_lora)
scores_fft,  sent_bleus_fft  = compute_bleu_scores(hyps_fft,  refs_fft)

print('=== Exp 3 LoRA ===')
for k, v in scores_lora.items():
    print(f'  {k:<25} {v}')
print('\n=== Exp 3 FFT ===')
for k, v in scores_fft.items():
    print(f'  {k:<25} {v}')


## 4. BERTScore (roberta-large)

Using `roberta-large` here (vs `distilbert-base-uncased` in exp2) for higher-quality semantic similarity scores. Requires ~1.4 GB extra VRAM.

In [ ]:
print('Computing BERTScore with roberta-large (LoRA) ...')
bert_lora = compute_bert_scores(hyps_lora, refs_lora, model_type='roberta-large')
print('LoRA BERTScore (roberta-large):', bert_lora)

print('\nComputing BERTScore with roberta-large (FFT) ...')
bert_fft  = compute_bert_scores(hyps_fft,  refs_fft,  model_type='roberta-large')
print('FFT BERTScore (roberta-large): ', bert_fft)


## 5. Save Scores

In [ ]:
all_scores = {
    'lora': {**scores_lora, **bert_lora},
    'fft':  {**scores_fft,  **bert_fft},
    'bert_model': 'roberta-large',
}
with open(RESULTS_DIR / 'bleu_scores.json', 'w') as f:
    json.dump(all_scores, f, indent=2)
print('Scores saved to outputs/exp3/results/bleu_scores.json')
print(json.dumps(all_scores, indent=2))


## 6. Sentence BLEU Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, scores, label, color in zip(
    axes,
    [sent_bleus_lora, sent_bleus_fft],
    ['LoRA (Exp 3, r=32, Uni)', 'FFT (Exp 3, 1.5B, Uni)'],
    ['steelblue', 'darkorange']
):
    ax.hist(scores, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(scores), color='red', linestyle='--', label=f'Mean = {np.mean(scores):.1f}')
    ax.axvline(np.median(scores), color='green', linestyle=':', label=f'Median = {np.median(scores):.1f}')
    ax.set_title(f'Sentence BLEU — {label}')
    ax.set_xlabel('Sentence BLEU'); ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'exp3_bleu_distributions.png', dpi=150)
plt.show()
